# Model Ensemble
> Ensemble of baseline work

In [ ]:
import os
import pandas as pd
import numpy as np
import gc
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras import initializers
from tensorflow import keras
import mlcrate as mlc
import pickle as pkl
from tensorflow.keras.layers import BatchNormalization
from keras.models import Sequential, Model
from keras.layers import Input, Embedding, Dense, Flatten, Concatenate, Dot, Reshape, Add, Subtract
from keras import backend as K
from keras import regularizers 
from tensorflow.keras.optimizers import Adam
from keras.callbacks import EarlyStopping, ModelCheckpoint
from keras.regularizers import l2
from sklearn.base import clone
from typing import Dict
from scipy import stats
from tensorflow.keras.losses import Loss
from tensorflow.keras import backend as K
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import TimeSeriesSplit, StratifiedKFold, KFold, GroupKFold
from tqdm import tqdm
from tensorflow.python.ops import math_ops

## 1. Data Exploration & Features layers

In [ ]:
%%time
n_features = 300
features = [f'f_{i}' for i in range(n_features)]
feature_columns = ['investment_id', 'time_id'] + features
train = pd.read_pickle('../input/ubiquant-market-prediction-half-precision-pickle/train.pkl')
train.head()

In [ ]:
investment_id = train.pop("investment_id")
investment_id.head()

In [ ]:
_ = train.pop("time_id")
y = train.pop("target")
y.head()

In [ ]:
%%time
investment_ids = list(investment_id.unique())
investment_id_size = len(investment_ids) + 1
investment_id_lookup_layer = layers.IntegerLookup(max_tokens=investment_id_size)
with tf.device("cpu"):
    investment_id_lookup_layer.adapt(investment_id)

In [ ]:
investment_id2 = investment_id[~investment_id.isin([85, 905, 2558, 3662, 2800, 1415])]

investment_ids2 = list(investment_id2.unique())
investment_id_size2 = len(investment_ids2) + 1
investment_id_lookup_layer2 = layers.IntegerLookup(max_tokens=investment_id_size2)
investment_id_lookup_layer2.adapt(pd.DataFrame({"investment_ids":investment_ids}))

In [ ]:
def preprocess(X, y):
    print(X)
    print(y)
    return X, y
def make_dataset(feature, investment_id, y, batch_size=1024, mode="train"):
    ds = tf.data.Dataset.from_tensor_slices(((investment_id, feature), y))
    ds = ds.map(preprocess)
    if mode == "train":
        ds = ds.shuffle(256)
    ds = ds.batch(batch_size).cache().prefetch(tf.data.experimental.AUTOTUNE)
    return ds

### feature_time_ds

In [ ]:
def make_ft_dataset(investment_id, feature, time_id, y=None, batch_size=1024):
    if y is not None:
        slices = ((investment_id, feature, time_id), y)
    else:
        slices = ((investment_id, feature, time_id))
        
    ds = tf.data.Dataset.from_tensor_slices(slices)
    ds = ds.batch(batch_size).cache().prefetch(tf.data.experimental.AUTOTUNE)
    return ds

## 2. DNN Architecture

In [ ]:
class MyModel(keras.Model):
    
    def __init__(self, investment_id, device='gpu'):
        super().__init__()
        
        with tf.device(device):
            self.inv_embedding = layers.Embedding(investment_id_size, 32)
            self.inv_fc = keras.Sequential([
                layers.Dense(64, activation='swish', kernel_initializer='he_normal', bias_initializer='zeros'),
                layers.Dropout(0.5),
                layers.Dense(32, activation='swish', kernel_initializer='he_normal', bias_initializer='zeros'),
                layers.Dropout(0.5),
            ])

            self.fea_fc = keras.Sequential([
                layers.Dense(256, activation='swish', kernel_initializer='he_normal', bias_initializer='zeros'),
                keras.layers.BatchNormalization(axis=1),
                layers.Dropout(0.5),
                layers.Dense(128, activation='swish', kernel_initializer='he_normal', bias_initializer='zeros'),
                keras.layers.BatchNormalization(axis=1),
                layers.Dropout(0.5),
                layers.Dense(64, activation='swish', kernel_initializer='he_normal', bias_initializer='zeros')
            ])
            
            self.fc = keras.Sequential([
                layers.Dropout(0.5),
                layers.Dense(128, activation='swish', kernel_initializer='he_normal', bias_initializer='zeros', kernel_regularizer="l2"),
                layers.Dropout(0.5),
                layers.Dense(32, activation='swish', kernel_initializer='he_normal', bias_initializer='zeros',  kernel_regularizer="l2"),
                layers.Dropout(0.5),
                layers.Dense(16, activation='swish', kernel_initializer='he_normal', bias_initializer='zeros', kernel_regularizer="l2"),
                layers.Dense(1)
            ])
    
    def call(self, inputs):
        inv_id, fea = inputs
        
        inv = investment_id_lookup_layer(inv_id)
        inv = self.inv_embedding(inv)
        inv = self.inv_fc(inv)
        inv = tf.squeeze(inv, axis=1)
        
        fea = self.fea_fc(fea)
        
        concat = tf.concat([inv, fea], axis=1)
        output = self.fc(concat)
        
        return output

In [ ]:
def get_model():
    investment_id_inputs = tf.keras.Input((1, ), dtype=tf.uint16)
    features_inputs = tf.keras.Input((300, ), dtype=tf.float16)
    
    investment_id_x = investment_id_lookup_layer(investment_id_inputs)
    investment_id_x = layers.Embedding(investment_id_size, 32, input_length=1)(investment_id_x)
    investment_id_x = layers.Reshape((-1, ))(investment_id_x)
    investment_id_x = layers.Dense(64, activation='swish')(investment_id_x)
    investment_id_x = layers.Dense(64, activation='swish')(investment_id_x)
    investment_id_x = layers.Dense(64, activation='swish')(investment_id_x)
    
    feature_x = layers.Dense(256, activation='swish')(features_inputs)
    feature_x = layers.Dense(256, activation='swish')(feature_x)
    feature_x = layers.Dense(256, activation='swish')(feature_x)
    
    x = layers.Concatenate(axis=1)([investment_id_x, feature_x])
    x = layers.Dense(512, activation='swish', kernel_regularizer="l2")(x)
    x = layers.Dense(128, activation='swish', kernel_regularizer="l2")(x)
    x = layers.Dense(32, activation='swish', kernel_regularizer="l2")(x)
    output = layers.Dense(1)(x)
    rmse = keras.metrics.RootMeanSquaredError(name="rmse")
    model = tf.keras.Model(inputs=[investment_id_inputs, features_inputs], outputs=[output])
    model.compile(optimizer=tf.optimizers.Adam(0.001), loss='mse', metrics=['mse', "mae", "mape", rmse])
    return model


def get_model2():
    investment_id_inputs = tf.keras.Input((1, ), dtype=tf.uint16)
    features_inputs = tf.keras.Input((300, ), dtype=tf.float16)
    
    investment_id_x = investment_id_lookup_layer(investment_id_inputs)
    investment_id_x = layers.Embedding(investment_id_size, 32, input_length=1)(investment_id_x)
    investment_id_x = layers.Reshape((-1, ))(investment_id_x)
    investment_id_x = layers.Dense(64, activation='swish')(investment_id_x)    
    investment_id_x = layers.Dense(64, activation='swish')(investment_id_x)
    investment_id_x = layers.Dense(64, activation='swish')(investment_id_x)
    investment_id_x = layers.Dense(64, activation='swish')(investment_id_x)
   # investment_id_x = layers.Dropout(0.65)(investment_id_x)
   
    
    feature_x = layers.Dense(256, activation='swish')(features_inputs)
    feature_x = layers.Dense(256, activation='swish')(feature_x)
    feature_x = layers.Dense(256, activation='swish')(feature_x)
    feature_x = layers.Dense(256, activation='swish')(feature_x)
    feature_x = layers.Dropout(0.65)(feature_x)
    
    x = layers.Concatenate(axis=1)([investment_id_x, feature_x])
    x = layers.Dense(512, activation='swish', kernel_regularizer="l2")(x)
   # x = layers.Dropout(0.2)(x)
    x = layers.Dense(128, activation='swish', kernel_regularizer="l2")(x)
  #  x = layers.Dropout(0.4)(x)
    x = layers.Dense(32, activation='swish', kernel_regularizer="l2")(x)
    x = layers.Dense(32, activation='swish', kernel_regularizer="l2")(x)
    x = layers.Dropout(0.75)(x)
    output = layers.Dense(1)(x)
    rmse = keras.metrics.RootMeanSquaredError(name="rmse")
    model = tf.keras.Model(inputs=[investment_id_inputs, features_inputs], outputs=[output])
    model.compile(optimizer=tf.optimizers.Adam(0.001), loss='mse', metrics=['mse', "mae", "mape", rmse])
    return model


def get_model5():
    features_inputs = tf.keras.Input((300, ), dtype=tf.float16)
    
    ## feature ##
    feature_x = layers.Dense(256, activation='swish')(features_inputs)
    feature_x = layers.Dropout(0.1)(feature_x)
    ## convolution 1 ##
    feature_x = layers.Reshape((-1,1))(feature_x)
    feature_x = layers.Conv1D(filters=16, kernel_size=4, strides=1, padding='same')(feature_x)
    feature_x = layers.BatchNormalization()(feature_x)
    feature_x = layers.LeakyReLU()(feature_x)
    ## convolution 2 ##
    feature_x = layers.Conv1D(filters=16, kernel_size=4, strides=4, padding='same')(feature_x)
    feature_x = layers.BatchNormalization()(feature_x)
    feature_x = layers.LeakyReLU()(feature_x)
    ## convolution 3 ##
    feature_x = layers.Conv1D(filters=64, kernel_size=4, strides=1, padding='same')(feature_x)
    feature_x = layers.BatchNormalization()(feature_x)
    feature_x = layers.LeakyReLU()(feature_x)
    ## convolution 4 ##
    feature_x = layers.Conv1D(filters=64, kernel_size=4, strides=4, padding='same')(feature_x)
    feature_x = layers.BatchNormalization()(feature_x)
    feature_x = layers.LeakyReLU()(feature_x)
    ## convolution 5 ##
    feature_x = layers.Conv1D(filters=64, kernel_size=4, strides=2, padding='same')(feature_x)
    feature_x = layers.BatchNormalization()(feature_x)
    feature_x = layers.LeakyReLU()(feature_x)
    ## flatten ##
    feature_x = layers.Flatten()(feature_x)
    
    x = layers.Dense(512, activation='swish', kernel_regularizer="l2")(feature_x)
    
    x = layers.Dropout(0.1)(x)
    x = layers.Dense(128, activation='swish', kernel_regularizer="l2")(x)
    x = layers.Dropout(0.1)(x)
    x = layers.Dense(32, activation='swish', kernel_regularizer="l2")(x)
    x = layers.Dropout(0.1)(x)
    output = layers.Dense(1)(x)
    rmse = keras.metrics.RootMeanSquaredError(name="rmse")
    model = tf.keras.Model(inputs=[features_inputs], outputs=[output])
    model.compile(optimizer=tf.optimizers.Adam(0.001), loss='mse', metrics=['mse', "mae", "mape", rmse])
    return model
del train
# del investment_id
del y
gc.collect()

## 2.0 Model_Dropout_10_RMSE

In [ ]:
def get_model_dr04():
    features_inputs = tf.keras.Input((300, ), dtype=tf.float32)
    
    feature_x = layers.Dense(256, activation='swish')(features_inputs)
    feature_x = layers.Dropout(0.4)(feature_x)
    feature_x = layers.Dense(128, activation='swish')(feature_x)
    feature_x = layers.Dropout(0.4)(feature_x)
    feature_x = layers.Dense(64, activation='swish')(feature_x)
    
    x = layers.Concatenate(axis=1)([feature_x])
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(64, activation='swish', kernel_regularizer="l2")(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(32, activation='swish', kernel_regularizer="l2")(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(16, activation='swish', kernel_regularizer="l2")(x)
    x = layers.Dropout(0.4)(x)
    output = layers.Dense(1)(x)
    output = tf.keras.layers.BatchNormalization(axis=1)(output)
    rmse = keras.metrics.RootMeanSquaredError(name="rmse")
    model = tf.keras.Model(inputs=[features_inputs], outputs=[output])
    model.compile(optimizer=tf.optimizers.Adam(0.001),  loss = correlationLoss, metrics=[correlationMetric])
    return model

dr=0.3

gpus = tf.config.experimental.list_physical_devices('GPU')
for gpu in gpus:
    print("Name:", gpu.name, "  Type:", gpu.device_type)

n_features = 300
features = [f'f_{i}' for i in range(n_features)]

# def preprocess(X, y):
#     return X, y
# def make_dataset(feature, y, batch_size=1024, mode="train"):
#     ds = tf.data.Dataset.from_tensor_slices((feature, y))
#     ds = ds.map(preprocess)
#     if mode == "train":
#         ds = ds.shuffle(512)
# #     ds = ds.batch(batch_size).cache().prefetch(tf.data.experimental.AUTOTUNE)
#     ds = ds.batch(batch_size).prefetch(tf.data.experimental.AUTOTUNE)
#     return ds

def correlationMetric(x, y, axis=-2):
  """Metric returning the Pearson correlation coefficient of two tensors over some axis, default -2."""
  x = tf.convert_to_tensor(x)
  y = math_ops.cast(y, x.dtype)
  n = tf.cast(tf.shape(x)[axis], x.dtype)
  xsum = tf.reduce_sum(x, axis=axis)
  ysum = tf.reduce_sum(y, axis=axis)
  xmean = xsum / n
  ymean = ysum / n
  xvar = tf.reduce_sum( tf.math.squared_difference(x, xmean), axis=axis)
  yvar = tf.reduce_sum( tf.math.squared_difference(y, ymean), axis=axis)
  cov = tf.reduce_sum( (x - xmean) * (y - ymean), axis=axis)
  corr = cov / tf.sqrt(xvar * yvar)
  return tf.constant(1.0, dtype=x.dtype) - corr


def correlationLoss(x,y, axis=-2):
  """Loss function that maximizes the pearson correlation coefficient between the predicted values and the labels,
  while trying to have the same mean and variance"""
  x = tf.convert_to_tensor(x)
  y = math_ops.cast(y, x.dtype)
  n = tf.cast(tf.shape(x)[axis], x.dtype)
  xsum = tf.reduce_sum(x, axis=axis)
  ysum = tf.reduce_sum(y, axis=axis)
  xmean = xsum / n
  ymean = ysum / n
  xsqsum = tf.reduce_sum( tf.math.squared_difference(x, xmean), axis=axis)
  ysqsum = tf.reduce_sum( tf.math.squared_difference(y, ymean), axis=axis)
  cov = tf.reduce_sum( (x - xmean) * (y - ymean), axis=axis)
  corr = cov / tf.sqrt(xsqsum * ysqsum)
  return tf.convert_to_tensor( K.mean(tf.constant(1.0, dtype=x.dtype) - corr ) , dtype=tf.float32 )
def correlationMetric_01mse(x, y, axis=-2):
  """Metric returning the Pearson correlation coefficient of two tensors over some axis, default -2."""
  x = tf.convert_to_tensor(x)
  y = math_ops.cast(y, x.dtype)
  n = tf.cast(tf.shape(x)[axis], x.dtype)
  xsum = tf.reduce_sum(x, axis=axis)
  ysum = tf.reduce_sum(y, axis=axis)
  xmean = xsum / n
  ymean = ysum / n
  xvar = tf.reduce_sum( tf.math.squared_difference(x, xmean), axis=axis)
  yvar = tf.reduce_sum( tf.math.squared_difference(y, ymean), axis=axis)
  cov = tf.reduce_sum( (x - xmean) * (y - ymean), axis=axis)
  corr = cov / tf.sqrt(xvar * yvar)
  return tf.constant(1.0, dtype=x.dtype) - corr

def correlation(x, y, axis=-2):
    xmean = tf.reduce_mean(x, axis=axis)
    ymean = tf.reduce_mean(y, axis=axis)
    cossim = keras.losses.cosine_similarity(x - xmean, y - ymean, axis=axis)
    return 1 + cossim

gc.collect()

# list(GroupKFold(5).split(train , groups = train.index))[0]
def pearson_coef(data):
    return data.corr()['target']['preds']

def evaluate_metric(valid_df):
    return np.mean(valid_df[['time_id_', 'target', 'preds']].groupby('time_id').apply(pearson_coef))

In [ ]:
def get_model_best(ft_units, x_units, x_dropout):
    investment_id_inputs = tf.keras.Input((1, ), dtype=tf.uint16)
    features_inputs = tf.keras.Input((300, ), dtype=tf.float16)
    
    investment_id_x = investment_id_lookup_layer2(investment_id_inputs)
    investment_id_x = layers.Embedding(investment_id_size2, 32, input_length=1)(investment_id_x)
    investment_id_x = layers.Reshape((-1, ))(investment_id_x)
    investment_id_x = layers.Dense(128, activation='swish')(investment_id_x)
    investment_id_x = layers.Dense(128, activation='swish')(investment_id_x)
    investment_id_x = layers.Dense(128, activation='swish')(investment_id_x)
    
    feature_x = layers.Dense(256, activation='swish')(features_inputs)
    for hu in ft_units:
        feature_x = layers.Dense(hu, activation='swish')(feature_x)
    
    x = layers.Concatenate(axis=1)([investment_id_x, feature_x])
    
    for i in range(len(x_units)):
        x = tf.keras.layers.Dense(x_units[i], kernel_regularizer="l2")(x) #v8
        x = tf.keras.layers.BatchNormalization()(x) #v7
        x = tf.keras.layers.Activation('swish')(x) #v7
        x = tf.keras.layers.Dropout(x_dropout[i])(x) #v8
        
    output = layers.Dense(1)(x)
    rmse = keras.metrics.RootMeanSquaredError(name="rmse")
    model = tf.keras.Model(inputs=[investment_id_inputs, features_inputs], outputs=[output])
    model.compile(optimizer=tf.optimizers.Adam(0.0001), loss='mse', metrics=['mse', "mae", "mape", rmse])
    return model

params = {
    'ft_units': [256,256],
    'x_units': [512, 256, 128, 32],
    'x_dropout': [0.4, 0.3, 0.2, 0.1]
#           'lr':1e-3, 
         }

models_best = []
scores = []
for i in range(7):
    model = get_model_best(**params)
    model.load_weights(f"../input/wmodels/best/model_{i}.tf")
    models_best.append(model)

## 2.1 Augment Model: Gaussian_Conv1 + Conv2d Model: Account for Spatial/Area Relationship

In [ ]:
def get_model6():
    features_inputs = tf.keras.Input((300, ), dtype=tf.float16)
    
    features_x = layers.GaussianNoise(0.1)(features_inputs)
    ## feature ##
    feature_x = layers.Dense(256, activation='swish')(features_x)
    feature_x = layers.Dropout(0.1)(feature_x)
    ## convolution 1 ##
    feature_x = layers.Reshape((-1,1))(feature_x)
    feature_x = layers.Conv1D(filters=16, kernel_size=4, strides=1, padding='same')(feature_x)
    feature_x = layers.BatchNormalization()(feature_x)
    feature_x = layers.LeakyReLU()(feature_x)
    ## convolution 2 ##
    feature_x = layers.Conv1D(filters=16, kernel_size=4, strides=4, padding='same')(feature_x)
    feature_x = layers.BatchNormalization()(feature_x)
    feature_x = layers.LeakyReLU()(feature_x)
    ## convolution 3 ##
    feature_x = layers.Conv1D(filters=64, kernel_size=4, strides=1, padding='same')(feature_x)
    feature_x = layers.BatchNormalization()(feature_x)
    feature_x = layers.LeakyReLU()(feature_x)
    ## convolution 4 ##
    feature_x = layers.Conv1D(filters=64, kernel_size=4, strides=4, padding='same')(feature_x)
    feature_x = layers.BatchNormalization()(feature_x)
    feature_x = layers.LeakyReLU()(feature_x)
    ## convolution 5 ##
    feature_x = layers.Conv1D(filters=64, kernel_size=4, strides=2, padding='same')(feature_x)
    feature_x = layers.BatchNormalization()(feature_x)
    feature_x = layers.LeakyReLU()(feature_x)
    ## flatten ##
    feature_x = layers.Flatten()(feature_x)
 
    x = layers.Dense(512, activation='swish', kernel_regularizer="l2")(feature_x)
    
    x = layers.Dropout(0.1)(x)
    x = layers.Dense(128, activation='swish', kernel_regularizer="l2")(x)
    x = layers.Dropout(0.1)(x)
    x = layers.Dense(32, activation='swish', kernel_regularizer="l2")(x)
    x = layers.Dropout(0.1)(x)
    output = layers.Dense(1)(x)
    rmse = keras.metrics.RootMeanSquaredError(name="rmse")
    model = tf.keras.Model(inputs=[features_inputs], outputs=[output])
    model.compile(optimizer=tf.optimizers.Adam(0.001), loss='mse', metrics=['mse', "mae", "mape", rmse])
    return model

def get_model7():
    features_inputs = tf.keras.Input((300, ), dtype=tf.float16)
    
    ## Dense 1 ##
    feature_x = layers.Dense(256, activation='swish')(features_inputs)
    feature_x = layers.Dropout(0.1)(feature_x)
    ## convolution 1 ##
    feature_x = layers.Reshape((-1,1))(feature_x)
    feature_x = layers.Conv1D(filters=16, kernel_size=4, strides=1, padding='same')(feature_x)
    feature_x = layers.BatchNormalization()(feature_x)
    feature_x = layers.LeakyReLU()(feature_x)
    ## convolution 2 ##
    feature_x = layers.Conv1D(filters=16, kernel_size=4, strides=4, padding='same')(feature_x)
    feature_x = layers.BatchNormalization()(feature_x)
    feature_x = layers.LeakyReLU()(feature_x)
    ## convolution 3 ##
    feature_x = layers.Conv1D(filters=64, kernel_size=4, strides=1, padding='same')(feature_x)
    feature_x = layers.BatchNormalization()(feature_x)
    feature_x = layers.LeakyReLU()(feature_x)

    ## convolution2D 1 ##
    feature_x = layers.Reshape((64,64,1))(feature_x)
    feature_x = layers.Conv2D(filters=32, kernel_size=4, strides=1, padding='same')(feature_x)
    feature_x = layers.BatchNormalization()(feature_x)
    feature_x = layers.LeakyReLU()(feature_x)
    ## convolution2D 2 ##
    feature_x = layers.Conv2D(filters=32, kernel_size=4, strides=4, padding='same')(feature_x)
    feature_x = layers.BatchNormalization()(feature_x)
    feature_x = layers.LeakyReLU()(feature_x)
    ## convolution2D 3 ##
    feature_x = layers.Conv2D(filters=32, kernel_size=4, strides=4, padding='same')(feature_x)
    feature_x = layers.BatchNormalization()(feature_x)
    feature_x = layers.LeakyReLU()(feature_x)

    ## flatten ##
    feature_x = layers.Flatten()(feature_x)
    ## Dense 3 ##
    x = layers.Dense(512, activation='swish', kernel_regularizer="l2")(feature_x)
    ## Dense 4 ##
    x = layers.Dropout(0.1)(x)
    ## Dense 5 ##    
    x = layers.Dense(128, activation='swish', kernel_regularizer="l2")(x)
    x = layers.Dropout(0.1)(x)
    ## Dense 6 ##
    x = layers.Dense(32, activation='swish', kernel_regularizer="l2")(x)
    x = layers.Dropout(0.1)(x)
    ## Dense 7 ##
    output = layers.Dense(1)(x)
    rmse = keras.metrics.RootMeanSquaredError(name="rmse")
    model = tf.keras.Model(inputs=[features_inputs], outputs=[output])
    model.compile(optimizer=tf.optimizers.Adam(0.001), loss='mse', metrics=['mse', "mae", "mape", rmse])
    return model

## 2.2 Augment2: Feature_Time Model

In [ ]:
def get_model_ft():
    investment_id_input = tf.keras.Input(shape=(1,), dtype=tf.uint16, name='investment_id')
    inv_x = layers.Dense(64, activation='relu')(investment_id_input)
    inv_x = layers.Dropout(0.2)(inv_x)

    features_input = tf.keras.Input(shape=(300,), dtype=tf.float16, name='features')
    f_x = layers.Dense(512, activation='relu')(features_input)
    f_x = layers.Dropout(0.25)(f_x)
    f_x = layers.Dense(256, activation='relu')(f_x)
    f_x = layers.Dropout(0.2)(f_x)

    time_id_input = tf.keras.Input(shape=(1,), dtype=tf.uint16, name='time_id')
    time_x = layers.Dense(64, activation='relu')(time_id_input)
    time_x = layers.Dropout(0.2)(time_x)

    concatenated = layers.concatenate([inv_x, f_x, time_x], axis=-1)
    output = layers.Dense(1)(concatenated)

    model = tf.keras.models.Model([investment_id_input, features_input, time_id_input], output, name='model_with_time_id')
    
    model.compile(optimizer='rmsprop', loss='mse', metrics=['mse', 'mae', 'mape'])
    return model

In [ ]:
gc.collect()
model_ft = get_model_ft()
model_ft.summary()

In [ ]:
models = []

for i in range(5):
    model = get_model()
    model.load_weights(f'../input/dnn-base/model_{i}')
    models.append(model)

for i in range(10):
    model = get_model2()
    model.load_weights(f'../input/train-dnn-v2-10fold/model_{i}')
    models.append(model)

for i in range(5):
    model = MyModel(investment_id=investment_id, device='cpu')
    model.load_weights(f'../input/masked-model-weights/random_mask_DNN_ensemble_weights/model{i}/model_{i}.tf')
    models.append(model)
    
    
models2 = []
    
for i in range(5):
    model = get_model5()
    model.load_weights(f'../input/prediction-including-spatial-info-with-conv1d/model_{i}.tf')
    models2.append(model)
    
for i in range(5):
    model = get_model6()
    model.load_weights(f'../input/gaussian-noise-model-weights/model_{i}.tf')
    models2.append(model)

for i in range(5):
    model = get_model7()
    model.load_weights(f'../input/altconv2dweights/model_{i}.tf')
    models2.append(model)
    
models3 = []
    
for i in range(10):
    model = get_model_dr04()
    model.load_weights(f'../input/mse10-model-weights/model_{i}')
    models3.append(model)

model_ft = get_model_ft()
model_ft.load_weights(f'../input/feature-time-model/ns_model_with_time_id.tf')

# Transformer_masked_model

In [ ]:
def gen_ids_and_skf_idxs():
    train = pd.read_pickle('../input/ubiquant-market-prediction-half-precision-pickle/train.pkl')
    investment_id = train[["investment_id"]].astype('int64')
    train.pop("investment_id")
    train.pop("time_id")
    train.pop("target")
    skf = StratifiedKFold(5, shuffle=True, random_state=42)
    idxs = list(enumerate(skf.split(train, investment_id)))
    del train
    gc.collect()
    return investment_id, idxs

In [ ]:
'''investment_id, idxs = gen_ids_and_skf_idxs()

investment_ids = list(np.unique(investment_id.values))
investment_id_size = len(investment_ids) + 1
id_lookup_layer = layers.IntegerLookup(max_tokens=investment_id_size)
id_lookup_layer.adapt(investment_id)'''

In [ ]:
class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        self.att = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = keras.Sequential(
            [layers.Dense(ff_dim, activation="relu"), layers.Dense(embed_dim),]
        )
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(rate)
        self.dropout2 = layers.Dropout(rate)

    def call(self, inputs, training):
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

In [ ]:
def get_model_transformer():
  embed_dim = 300  # Embedding size for each token
  num_heads = 8  # Number of attention heads
  ff_dim = 512  # Hidden layer size in feed forward network inside transformer

  transformer_block = TransformerBlock(embed_dim, num_heads, ff_dim)
  features_input = tf.keras.Input((300,), dtype=tf.float32)
  investment_id_input = tf.keras.Input((1,))

  investment_id_x = investment_id_lookup_layer(investment_id_input)
  investment_id_x = layers.Embedding(investment_id_size, 32)(investment_id_x)
  investment_id_x = layers.Dense(64, activation='swish', kernel_initializer='he_normal', bias_initializer='zeros')(investment_id_x)
  investment_id_x = layers.Dropout(0.5)(investment_id_x)
  investment_id_x = layers.Dense(32, activation='swish', kernel_initializer='he_normal', bias_initializer='zeros')(investment_id_x)
  investment_id_x = layers.Dropout(0.5)(investment_id_x)
  investment_id_x = tf.squeeze(investment_id_x, axis=1)

  features_inputs = tf.expand_dims(investment_id_x, axis = -1) + tf.expand_dims(features_input, axis = 1)
  # (None, 32, 300)

# 6 transformer blocks
  features_x = transformer_block(features_inputs)

  features_x = layers.Reshape((-1,))(features_x)
# MLP for output
  output_x = layers.Dense(512, activation='swish',
                          kernel_initializer=initializers.RandomNormal(stddev=1))(features_x)
  output_x = layers.Dropout(0.6)(output_x)
  output_x = layers.Dense(256, activation='swish',
                          kernel_initializer=initializers.RandomNormal(stddev=1))(output_x)
  output_x = layers.Dropout(0.6)(output_x)
  output_x = layers.Dense(128, activation='swish',
                          kernel_initializer=initializers.RandomNormal(stddev=1))(output_x)
  output_x = layers.Dense(64, activation='swish',
                          kernel_initializer=initializers.RandomNormal(stddev=1),
                          kernel_regularizer="l2")(output_x)
  output_x = layers.Dropout(0.6)(output_x)
  output_x = layers.Dense(32, activation='swish',
                          kernel_initializer=initializers.RandomNormal(stddev=1),
                          kernel_regularizer="l2")(output_x)
  output_x = layers.Dropout(0.6)(output_x)
  output_x = layers.Dense(16, activation='swish',
                          kernel_initializer=initializers.RandomNormal(stddev=1),
                          kernel_regularizer="l2")(output_x)
  output_x = layers.Dropout(0.6)(output_x)
  output = layers.Dense(1)(output_x)
  output = tf.keras.layers.BatchNormalization(axis=1)(output)

# Model compilation
  rmse = keras.metrics.RootMeanSquaredError(name="rmse")
  model = tf.keras.Model(inputs=[investment_id_input, features_input], outputs=[output])
  # learning rate decay in Adam
  initial_learning_rate = 5e-4
  lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate,
        decay_steps=10000,
        decay_rate=0.9,
        staircase=True)
  model.compile(optimizer=tf.optimizers.Adam(learning_rate=lr_schedule), 
                loss='mse', 
                metrics=[rmse, correlation])
  return model

# End of transformer_masked model

In [ ]:
def get_model_corr(ft_units, x_units, x_dropout):
    
    # investment_id
    investment_id_inputs = tf.keras.Input((1, ), dtype=tf.uint16)
    investment_id_x = investment_id_lookup_layer(investment_id_inputs)
    investment_id_x = layers.Embedding(investment_id_size, 32, input_length=1)(investment_id_x)
    investment_id_x = layers.Reshape((-1, ))(investment_id_x)
    investment_id_x = layers.Dense(128, activation='swish')(investment_id_x)
    investment_id_x = layers.Dense(128, activation='swish')(investment_id_x) 
    investment_id_x = layers.Dense(128, activation='swish')(investment_id_x)
    
    # features_inputs
    features_inputs = tf.keras.Input((300, ), dtype=tf.float16)
    bn = tf.keras.layers.BatchNormalization()(features_inputs)
    gn = tf.keras.layers.GaussianNoise(0.035)(bn)
    feature_x = layers.Dense(300, activation='swish')(gn)
    feature_x = tf.keras.layers.Dropout(0.5)(feature_x)
    
    for hu in ft_units:
        feature_x = layers.Dense(hu, activation='swish')(feature_x)
#         feature_x = tf.keras.layers.Activation('swish')(feature_x)
        feature_x = tf.keras.layers.Dropout(0.35)(feature_x)
    
    x = layers.Concatenate(axis=1)([investment_id_x, feature_x])
    
    for i in range(len(x_units)):
        x = tf.keras.layers.Dense(x_units[i], kernel_regularizer="l2")(x) 
        x = tf.keras.layers.Activation('swish')(x)
        x = tf.keras.layers.Dropout(x_dropout[i])(x)
        
    output = layers.Dense(1)(x)
    model = tf.keras.Model(inputs=[investment_id_inputs, features_inputs], outputs=[output])
    model.compile(optimizer=tf.optimizers.Adam(0.0001), loss=correlationLoss, 
                  metrics=['mse', "mae", correlation])
    return model


params = {
#     'num_columns': len(features), 
    'ft_units': [150, 75, 150 ,200],
    'x_units': [512, 256, 128, 32],
    'x_dropout': [0.44, 0.4, 0.33, 0.2] #4, 3, 2, 1
#           'lr':1e-3, 
         }

# 3. Validation

In [ ]:
def preprocess_test(investment_id, feature):
    return (investment_id, feature), 0

def preprocess_test_s(feature):
    return (feature), 0

def make_test_dataset(feature, investment_id, batch_size=1024):
    ds = tf.data.Dataset.from_tensor_slices(((investment_id, feature)))
    ds = ds.map(preprocess_test)
    ds = ds.batch(batch_size).cache().prefetch(tf.data.experimental.AUTOTUNE)
    return ds

def make_test_dataset2(feature, batch_size=1024):
    ds = tf.data.Dataset.from_tensor_slices(((feature)))
    ds = ds.batch(batch_size).cache().prefetch(tf.data.AUTOTUNE)
    return ds

def inference(models, ds):
    y_preds = []
    for model in models:
        y_pred = model.predict(ds)
        y_preds.append(y_pred)
    return np.mean(y_preds, axis=0)

from sklearn.decomposition import PCA
pca = PCA(n_components=1)


def pca_inference(models, ds):
    y_preds = []
    for model in models:
        y_pred = model.predict(ds)
        y_preds.append(y_pred)
    res = np.hstack(y_preds)
    print(len(res))
    if len(res)>1:
        res = pca.fit_transform(res)
    else:
        res = np.mean(res, axis=1)
    return res

def make_test_dataset3(feature, batch_size=1024):
    ds = tf.data.Dataset.from_tensor_slices((feature))
    ds = ds.map(preprocess_test_s)
    ds = ds.batch(batch_size).cache().prefetch(tf.data.experimental.AUTOTUNE)
    return ds

def infer(models, ds):
    y_preds = []
    for model in models:
        y_pred = model.predict(ds)
        y_preds.append((y_pred-y_pred.mean())/y_pred.std())
    
    return np.mean(y_preds, axis=0)


In [ ]:
def inference_transformer(model, ds):
    y_preds = []
    for i in range(5):
        model.load_weights(f"../input/transformer-mask-weights/Transformer_mask_weights/model{i}/model_{i}.tf")  # private models' weights saved in training cell
        y_pred = model.predict(ds)
        y_preds.append(y_pred)
    return np.mean(y_preds, axis=0)

In [ ]:
import ubiquant
env = ubiquant.make_env()
iter_test = env.iter_test()

In [ ]:
 for (test_df, sample_prediction_df) in iter_test:
    ds = make_test_dataset(test_df[features], test_df["investment_id"])
    p1 = inference(models, ds) # DNN + masked_DNN combined
    ds2 = make_test_dataset2(test_df[features])
    p2 = inference(models2, ds2) # Conv2D
    ds3 = make_test_dataset3(test_df[features])
    p3 = infer(models3, ds3) # dropout = 0.4 for MLP, not used for this version
    p4 = inference(models_best, ds) # grid_search
    
    # feature_time_augment
    test_time_id = test_df['row_id'].str.split('_', expand=True).get(key=0).astype(int)
    ds5 = make_ft_dataset(investment_id=test_df['investment_id'], feature=test_df[features], time_id=test_time_id)
    p5 = model_ft.predict([test_df['investment_id'], test_df[features], test_time_id])[:, 0]
    
    # Transformer model
    ds_transformer = make_test_dataset(test_df[features], test_df["investment_id"].astype('int64'))
    transformer_model = get_model_transformer()
    p_t = inference_transformer(transformer_model, ds_transformer)
    
    sample_prediction_df['target'] = p1 * 0.18 + p2 * 0.57 + p3 * 0.07 + p_t * 0.07 + p5 * 0.11
    env.predict(sample_prediction_df) 
    display(sample_prediction_df)